# iTransformer Kaggle 학습 + 평가

이 노트북은 `itransformer_regime_corr_v1` 풀 학습을 Kaggle에서 실행하고, 학습 직후 evaluation artifact까지 `/kaggle/working`에 저장한다.

- 학습 artifact: `multi_horizon_model.keras`, `multi_horizon_scaler.pkl`, `metadata.json`
- 평가 artifact: signal, diagnostics, metric, objective, leaderboard CSV와 summary JSON
- 기본 정책: train `2021-01-01..2024-12-31`, eval 요청 구간 `2024-10-01..2024-12-31`, 2025 이후 holdout 보존

In [ ]:
import os

# Kaggle 입력 데이터셋과 출력 경로
os.environ.setdefault("PARQUET_DIR", "/kaggle/input/sisc-ai-trading-dataset")
os.environ.setdefault("OUTPUT_DIR", "/kaggle/working")
os.environ.setdefault("WEIGHTS_DIR", "/kaggle/working")

# 기간 정책: 2025 이후 데이터는 학습과 평가 label에 포함하지 않는다.
os.environ.setdefault("TRAIN_START_DATE", "2021-01-01")
os.environ.setdefault("TRAIN_END_DATE", "2024-12-31")
os.environ.setdefault("LABEL_CUTOFF_DATE", "2024-12-31")
os.environ.setdefault("EVAL_START_DATE", "2024-10-01")
os.environ.setdefault("EVAL_END_DATE", "2024-12-31")
os.environ.setdefault("HOLDOUT_START_DATE", "2025-01-01")

# 첫 풀 학습은 early stopping을 전제로 너무 길게 잡지 않는다.
os.environ.setdefault("SEQ_LEN", "60")
os.environ.setdefault("BATCH_SIZE", "32")
os.environ.setdefault("EPOCHS", "30")
os.environ.setdefault("EARLY_STOPPING_PATIENCE", "8")
os.environ.setdefault("REDUCE_LR_PATIENCE", "4")

print("PARQUET_DIR 후보=", os.environ["PARQUET_DIR"])
print("OUTPUT_DIR=", os.environ["OUTPUT_DIR"])
print("TRAIN=", os.environ["TRAIN_START_DATE"], "~", os.environ["TRAIN_END_DATE"])
print("EVAL=", os.environ["EVAL_START_DATE"], "~", os.environ["EVAL_END_DATE"])
print("HOLDOUT_START=", os.environ["HOLDOUT_START_DATE"])

In [ ]:
from pathlib import Path
import glob
import sys
import zipfile

# upload_to_kaggle.py가 데이터셋에 포함한 코드 아카이브를 사용한다.
dataset_dir = Path(os.environ["PARQUET_DIR"])
price_candidates = [Path(path) for path in glob.glob("/kaggle/input/**/price_data.parquet", recursive=True)]
macro_candidates = [Path(path) for path in glob.glob("/kaggle/input/**/macroeconomic_indicators.parquet", recursive=True)]
if price_candidates:
    os.environ["PARQUET_DIR"] = str(price_candidates[0].parent)
    dataset_dir = price_candidates[0].parent
    print("자동 탐색 PARQUET_DIR=", dataset_dir)
else:
    print("price_data.parquet 자동 탐색 실패. /kaggle/input 연결 데이터셋을 확인하세요.")

# 과거 실행 방식처럼 sisc-web 폴더가 직접 올라와 있으면 그 경로를 먼저 사용한다.
legacy_code_roots = [
    Path("/kaggle/working/sisc-web"),
    Path("/kaggle/input/sisc-web"),
]
legacy_code_roots.extend(Path(path).parent for path in glob.glob("/kaggle/input/**/AI/modules", recursive=True))
loaded_code_root = None
for code_root in legacy_code_roots:
    if (code_root / "AI" / "modules").exists():
        sys.path.insert(0, str(code_root))
        os.environ["SISC_WEB_ROOT"] = str(code_root)
        loaded_code_root = code_root
        print("코드 경로 로드:", code_root)
        break
code_archive_name = os.environ.get("KAGGLE_CODE_ARCHIVE_NAME", "sisc_ai_code.zip")
code_candidates = []
preferred_code_zip = dataset_dir / code_archive_name
if preferred_code_zip.exists():
    code_candidates.append(preferred_code_zip)
code_candidates.extend(Path(path) for path in glob.glob(f"/kaggle/input/**/{code_archive_name}", recursive=True))

if code_candidates:
    code_zip = code_candidates[0]
    code_root = Path("/kaggle/working/sisc_code")
    code_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(code_zip) as zf:
        zf.extractall(code_root)
    sys.path.insert(0, str(code_root))
    os.environ["SISC_WEB_ROOT"] = str(code_root)
    loaded_code_root = code_root
    print("코드 아카이브 로드:", code_zip)
else:
    print("코드 아카이브를 찾지 못했습니다. 이미 코드 경로가 로드됐는지 다음 셀의 import로 확인합니다.")

In [ ]:
from pathlib import Path
import json

parquet_dir = Path(os.environ["PARQUET_DIR"])
required_inputs = ["price_data.parquet", "macroeconomic_indicators.parquet"]
missing_inputs = [name for name in required_inputs if not (parquet_dir / name).exists()]
if missing_inputs:
    print("/kaggle/input 파일 탐색 결과:")
    for root, dirs, files in os.walk("/kaggle/input"):
        visible = [name for name in files if name.endswith((".parquet", ".zip", ".json"))]
        if visible:
            print(root, visible[:20])
    raise FileNotFoundError(f"필수 입력 parquet 누락: {missing_inputs}, 경로={parquet_dir}")

from AI.modules.signal.models.itransformer.kaggle_train_eval import smoke_config_summary

config_summary = smoke_config_summary()
print(json.dumps(config_summary, ensure_ascii=False, indent=2))

In [ ]:
import importlib.util
from pathlib import Path

# 예전 PatchTST 노트북처럼 파일 경로에서 직접 로드하는 방식을 우선 사용한다.
script_path = Path(os.environ.get("SISC_WEB_ROOT", "/kaggle/working/sisc-web")) / "AI/modules/signal/models/itransformer/kaggle_train_eval.py"
if script_path.exists():
    spec = importlib.util.spec_from_file_location("itransformer_kaggle_train_eval", script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    run_kaggle_train_eval = module.run_kaggle_train_eval
    print("학습+평가 스크립트 로드:", script_path)
else:
    from AI.modules.signal.models.itransformer.kaggle_train_eval import run_kaggle_train_eval
    print("학습+평가 스크립트 로드: Python package import")

# 실제 Kaggle GPU 학습과 학습 직후 평가를 연속 실행한다.
result = run_kaggle_train_eval()
eval_summary = result["eval_summary"]
print(json.dumps(eval_summary, ensure_ascii=False, indent=2))

In [ ]:
from pathlib import Path

working = Path(os.environ["OUTPUT_DIR"])
expected_outputs = [
    "multi_horizon_model.keras",
    "multi_horizon_scaler.pkl",
    "metadata.json",
    "itransformer_signal_frame.csv",
    "itransformer_diagnostics_frame.csv",
    "itransformer_metric_frame.csv",
    "itransformer_objective_frame.csv",
    "itransformer_leaderboard_frame.csv",
    "itransformer_eval_summary.json",
]

for name in expected_outputs:
    path = working / name
    if not path.exists():
        raise FileNotFoundError(f"기대 산출물 누락: {path}")
    print(f"{name}: {path.stat().st_size:,} bytes")

In [ ]:
import pandas as pd

leaderboard = pd.read_csv(Path(os.environ["OUTPUT_DIR"]) / "itransformer_leaderboard_frame.csv")
objective = pd.read_csv(Path(os.environ["OUTPUT_DIR"]) / "itransformer_objective_frame.csv")
display(leaderboard.head(20))
display(objective.head(20))